# F1 Race Analysis

Working notebook — grows alongside the analysis modules.
Each section validates a block against real race data.

**Race used:** 2023 Bahrain GP — clean race, no major weather events, strong teammate battles (VER/PER, HAM/RUS).

Sections:
1. Load & inspect raw lap data
2. Lap time volatility (Block 2)
3. Pairs / cointegration (Block 3)

In [ ]:
import sys
sys.path.insert(0, '..')  # so imports find data/ and analysis/ from the repo root

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

from data.fastf1_loader import get_accurate_laps, get_teammate_laps

## 1. Load race data

First run will download from the FastF1 API and cache locally — takes ~30s.  
Every run after that is instant from cache.

`get_accurate_laps` returns only `IsAccurate=True` laps — pit in/out laps and SC laps are already stripped.

In [ ]:
YEAR = 2023
EVENT = 'Bahrain'

laps = get_accurate_laps(YEAR, EVENT)

print(f"Total clean laps loaded: {len(laps)}")
print(f"Drivers: {sorted(laps['Driver'].unique())}")
laps.head()

## Lap times across the race — all drivers

Plotting every driver's lap times by lap number.  
What to look for:
- **Stint structure** — clusters of faster laps followed by a gap (pit stop) then faster laps again on fresh tyres
- **Outliers** — any anomalously slow laps that slipped through the `IsAccurate` filter
- **Backmarkers** — should sit visibly above the frontrunners

In [ ]:
drivers = sorted(laps['Driver'].unique())
colors = cm.tab20(np.linspace(0, 1, len(drivers)))

fig, ax = plt.subplots(figsize=(14, 6))

for driver, color in zip(drivers, colors):
    d = laps[laps['Driver'] == driver]
    ax.plot(d['LapNumber'], d['LapTime_s'], label=driver, color=color, linewidth=1, alpha=0.8)

ax.set_xlabel('Lap Number')
ax.set_ylabel('Lap Time (s)')
ax.set_title('2023 Bahrain GP — All drivers lap times')
ax.legend(ncol=4, fontsize=7, loc='upper right')
plt.tight_layout()
plt.show()

## 2. Lap time volatility

Which drivers are most consistent? How does tyre age affect variance?

- `driver_volatility_summary` ranks every driver by lap time std across the whole race — lower std = more consistent
- `stint_volatility` breaks it down per stint so you can see if a driver was erratic on one compound vs another
- Finance parallel: same as ranking stocks by realised volatility over a period, then decomposing by regime

In [ ]:
from analysis.lap_volatility import driver_volatility_summary, stint_volatility

# Overall consistency ranking — who had the most stable lap times across the whole race?
vol_summary = driver_volatility_summary(laps)
print("Driver consistency ranking (lower std = more consistent):")
print(vol_summary.to_string(index=False))

In [ ]:
# Volatility by stint — does a driver get more erratic as tyres wear?
# Each row is one driver's one stint: std tells you how much lap times varied within that stint
stint_vol = stint_volatility(laps)

# Focus on the top 6 drivers for readability
top6 = ['VER', 'PER', 'LEC', 'SAI', 'HAM', 'RUS']
print("Stint volatility — top 6 drivers:")
print(stint_vol[stint_vol['Driver'].isin(top6)].to_string(index=False))

In [ ]:
# Bar chart — driver consistency across the whole race
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(vol_summary['Driver'], vol_summary['std'], color='steelblue', alpha=0.8)
ax.set_xlabel('Driver')
ax.set_ylabel('Lap Time Std Dev (s)')
ax.set_title('2023 Bahrain GP — Driver consistency (lower = more consistent)')
ax.axhline(vol_summary['std'].mean(), color='red', linestyle='--', linewidth=1, label='Field average')
ax.legend()
plt.tight_layout()
plt.show()